In [1]:
import os
import glob
import json
import pandas as pd

In [2]:
root = '/home/daniel/Desktop/News-Retrieval-Explain-Challenge'
des_path = root + '/dict'

In [ ]:
import os

suffix = "_extract"
root_dir = "static/images/Keyframes"

for folder_name in os.listdir(root_dir):
    if folder_name.endswith(suffix):
        old_path = os.path.join(root_dir, folder_name)
        new_path = os.path.join(root_dir, folder_name[:-len(suffix)])
        os.rename(old_path, new_path)
        print(f"Renamed: {old_path} -> {new_path}")

# Part 1

In [3]:
data_root = root + '/static/images/Keyframes'
scene_root = root + '/dict/SceneJSON'

In [30]:
scene_id2info = dict()

for data_part in sorted(os.listdir(scene_root)):
    data_part_path = f'{scene_root}/{data_part}'
    scene_id2info[data_part] = dict()
    for video_path in sorted(os.listdir(data_part_path)):
        video_id = video_path.split('.')[0]
        video_path_full = f'{data_part_path}/{video_path}'
        video_fps = 30
        
        with open(video_path_full, 'r') as f:
            video_scene_info = json.load(f)
        
        scene_id2info[data_part][video_id] = {
            'lst_shot': dict()
        }
        
        for i, item in enumerate(video_scene_info):
            scene_id2info[data_part][video_id]['lst_shot'][str(i)] = {
                'shot_range': item,
                'shot_time': [item[0]/video_fps, item[1]/video_fps],
                'lst_keyframe_paths': [],
                'lst_keyframe_idxs': []
            }

In [8]:
scene_id2info['L21_a']['V001']['lst_shot']['1'].keys()

dict_keys(['shot_range', 'shot_time', 'lst_keyframe_paths', 'lst_keyframe_idxs'])

In [31]:
global_index = 0
id2img = []

for data_part in sorted(os.listdir(data_root)):
    data_part_path = f'{data_root}/{data_part}'

    for video_id in sorted(os.listdir(data_part_path)):
        video_dir = f'{data_part_path}/{video_id}'
        
        with open(f'{scene_root}/{data_part}/{video_id}.json', 'r') as f:
            video_scene_info = json.load(f)
        
        scene_track = 0
        for image_path in sorted(os.listdir(video_dir)):
            frame_idx = int(image_path.split('.')[0])
            
            image_path = f'{video_dir}/{image_path}'.replace(f'{root}', '')
            
            while(len(video_scene_info) > 0 and frame_idx > video_scene_info[0][1]):
                video_scene_info.pop(0)
                scene_track += 1
            
            if len(video_scene_info) == 0:
                continue
            
            info = {
                "image_path": image_path,
                "scene_idx": f'{data_part}/{video_id}/lst_shot/{str(scene_track)}'
            }
            
            scene_id2info[data_part][video_id]['lst_shot'][str(scene_track)]['lst_keyframe_paths'].append(image_path)
            scene_id2info[data_part][video_id]['lst_shot'][str(scene_track)]['lst_keyframe_idxs'].append(global_index)
            
            id2img.append(info)
            
            global_index += 1

In [32]:
id2img = dict(enumerate(id2img))

In [33]:
with open(f'{des_path}/scene_id2info.json', 'w') as f:
    f.write(json.dumps(scene_id2info))
    
with open(f'{des_path}/id2img.json', 'w') as f:
    f.write(json.dumps(id2img))

print(f"Number of Index: {len(id2img)}")

Number of Index: 97358


# Part 2

In [34]:
with open(f'{des_path}/scene_id2info.json', 'r') as f:
    SceneID2Info = json.load(f)
    
audios_detection_dir = f'{des_path}/audio_detection'


In [35]:
check_error = 0
audio_id2img_id = []
for data_part in sorted(os.listdir(audios_detection_dir)):
    for audio_detection_path in sorted(os.listdir(f'{audios_detection_dir}/{data_part}')):
        audio_id = audio_detection_path.replace('.json', '')
        scene_info = SceneID2Info[data_part][audio_id]['lst_shot']
        
        with open(f'{audios_detection_dir}/{data_part}/{audio_detection_path}', 'r') as f:
            audio_shots = json.load(f)
            
        i = 0
        scene_info_len = len(scene_info)
        for audio_interval in audio_shots:
            result = []
            start, end = audio_interval
                
            while True:
                if i >= scene_info_len:
                    break
                
                shot_interval = scene_info[str(i)]['shot_time']
                if end <= shot_interval[0]:
                    break
                if(start >= shot_interval[1]):
                    i += 1
                    continue
                    
                result.extend(scene_info[str(i)]['lst_keyframe_idxs'].copy()) 
                if end > shot_interval[1]:
                    i += 1
                    start = shot_interval[1]
                else:
                    break
                    
            audio_id2img_id.append(result)
            
            
            check_error += 1

In [36]:
with open(f'{des_path}/audio_id2img_id.json', 'w') as f:
    f.write(json.dumps(audio_id2img_id))

# Part 3

In [39]:
video_id2img_id = dict()

for data_part in SceneID2Info.keys():
    for video_id in SceneID2Info[data_part].keys():
        sample_key = f'{data_part}_{video_id}'
        video_id2img_id[sample_key] = []
        for key, value in SceneID2Info[data_part][video_id]['lst_shot'].items():
            video_id2img_id[sample_key].extend(value['lst_keyframe_idxs'])

In [43]:
with open(f'{des_path}/video_id2img_id.json', 'w') as f:
    f.write(json.dumps(video_id2img_id))

# Part 4

In [44]:
from tqdm import tqdm

def find_nearest(array, value):     
    array = np.asarray(array)
    idx = sorted((np.abs(array - value)).argsort()[:2].tolist())
    return idx

In [ ]:
audios_detection_dir = f'{des_path}/audio_detection'

audio_global_id = 0
img_id2audio_id = dict()
for data_part in tqdm(sorted(os.listdir(audios_detection_dir))):
    for audio_detection_path in sorted(os.listdir(f'{audios_detection_dir}/{data_part}')):
        audio_id = audio_detection_path.replace('.json', '')
        scene_info = SceneID2Info[data_part][audio_id]['lst_shot']
        
        with open(f'{audios_detection_dir}/{data_part}/{audio_detection_path}', 'r') as f:
            audio_shots = json.load(f)
        
        audio_pivot_shots = []
        for audio_shot in audio_shots:
            start, end = audio_shot
            audio_pivot_shots.append((start+end)/2)
        
        for shot in scene_info.values():
            shot_center = (shot['shot_time'][0] + shot['shot_time'][1])/2
            shot_frame_idxs = shot['lst_keyframe_idxs']
            nearest_audio = [audio_global_id + val for val in find_nearest(audio_pivot_shots, shot_center)]
            
            for shot_frame_idx in shot_frame_idxs:
                img_id2audio_id[shot_frame_idx] = nearest_audio
            
        audio_global_id += len(audio_shots)

In [ ]:
with open(f'{des_path}/img_id2audio_id.json', 'w') as f:
    f.write(json.dumps(img_id2audio_id))